###creating dataframe


In [0]:
df = spark.read.table(
  'learning.bigmart.sales'
)
display(df)

show schema

In [0]:
df.printSchema()

DDL schema 

In [0]:
myddlschema='''
| Item_Identifie / string;
| Item_Fat_Content / int;
| Item_Visibility / double;
| Item_Type / string;
| Item_MRP / double;
| Outlet_Identifier / string;
| Outlet_Establishment_Year / int;
| Outlet_Size / string;
| Outlet_Location_Type / string;
| Outlet_Type / string; 

'''

In [0]:
print(myddlschema)

In [0]:
# Read the table 'learning.bigmart.sales' into a DataFrame
df = spark.read.table('learning.bigmart.sales')

# Import necessary types for schema definition
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define the schema using StructType and StructField
myddlschema = StructType([
    StructField("Item_Identifie", StringType(), True),  # Define 'Item_Identifie' as StringType
    StructField("Item_Fat_Content", IntegerType(), True),  # Define 'Item_Fat_Content' as IntegerType
    StructField("Item_Visibility", DoubleType(), True),  # Define 'Item_Visibility' as DoubleType
    StructField("Item_Type", StringType(), True),  # Define 'Item_Type' as StringType
    StructField("Item_MRP", DoubleType(), True),  # Define 'Item_MRP' as DoubleType
    StructField("Outlet_Identifier", StringType(), True),  # Define 'Outlet_Identifier' as StringType
    StructField("Outlet_Establishment_Year", IntegerType(), True),  # Define 'Outlet_Establishment_Year' as IntegerType
    StructField("Outlet_Size", StringType(), True),  # Define 'Outlet_Size' as StringType
    StructField("Outlet_Location_Type", StringType(), True),  # Define 'Outlet_Location_Type' as StringType
    StructField("Outlet_Type", StringType(), True)  # Define 'Outlet_Type' as StringType
])

# Read the table 'learning.bigmart.sales' into a DataFrame with the defined schema
df = spark.read.schema(myddlschema).table('learning.bigmart.sales')

In [0]:
print(myddlschema)

###Selecting from table 

In [0]:
from pyspark.sql.functions import col
df=df.select(col('Item_Identifier'),col('Item_Fat_Content'))
display(df)

###Loading column with alias

In [0]:
from pyspark.sql.functions import col
df = spark.read.table('learning.bigmart.sales')
df = df.select(
    col('Item_Identifier').alias('Item_id'),
    col('Item_Fat_Content')
)
display(df)

###Filtering


In [0]:
df=df.filter(col('Item_Fat_Content')=='Low Fat')
display(df)

###Transforming a column by multiplying with a number

In [0]:
from pyspark.sql.functions import col
df=spark.read.table('learning.bigmart.sales')
df=df.withColumn('Item_Visibility',col('Item_Visibility')*10000)
df=df.select(
    col('Item_Identifier').alias('Item_id'),
    col('Item_Fat_Content'),
    col('Item_Visibility').alias('Item_Visibility')
)   
display(df)

###Overwrite table after updating table

In [0]:
spark.read.table('learning.bigmart.sales').write.mode('overwrite').saveAsTable('learning.bigmart.sales')

###New column with a constant 

In [0]:
from pyspark.sql.functions import lit, col

# Create the schema in the catalog named 'testing'
spark.sql("CREATE SCHEMA IF NOT EXISTS test")

# Read the existing table
df = spark.read.table('learning.bigmart.sales')

# Add the new column 'protien' with a default value
df = df.withColumn('protien', lit(0))

# Rename columns to match your new schema
df = df.select(
    col('Item_Identifier').alias('Item_id'),
    col('Item_Fat_Content').alias('Item_Fat'),
    col('Item_Visibility'),
    col('Item_Type'),
    col('Item_MRP').alias('Item_price'),
    col('Outlet_Identifier').alias('Outlet_Idr'),
    col('Outlet_Establishment_Year').alias('Outlet_Est_year'),
    col('Outlet_Size'),
    col('Outlet_Location_Type'),
    col('Outlet_Type'),
    col('protien')
)

# Write to the new table in the correct schema
df.write.mode('overwrite').saveAsTable('test.sales2')
display(df)

###Replacing column value 

In [0]:
from pyspark.sql.functions import regexp_replace, col

df = spark.read.table('test.sales2')
df = df.withColumn(
    'item_fat',
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col('item_fat'),
                "Low Fat",
                "LF"
            ),
            "low fat",
            "LF"
        ),
        "Regular",
        "R"
    )
)
df.write.mode('overwrite').saveAsTable('test.sales2')
display(df)

###Changing datatype

In [0]:
df=spark.read.table('test.sales2')
df = df.withColumn(
    'Outlet_Est_year',
    col('Outlet_Est_year').cast('string')
)
display(df)

###Sort in descening order

In [0]:

df=spark.read.table('test.sales2')
df = df.sort('Item_id',ascending=False)
display(df)


In [0]:
from pyspark.sql.functions import desc

df = spark.read.table('test.sales2')
df = df.sort(desc('Item_id'))
display(df)

###Creating a new Schema

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("Ctest1").getOrCreate()

# Define schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("Itemid", StringType(), True),
    StructField("Fat_Content", StringType(), True),
    StructField("Visibility", DoubleType(), True),
    StructField("Type", StringType(), True),
    StructField("Price", DoubleType(), True)
])

# Create an empty DataFrame with the defined schema
df = spark.createDataFrame([], schema)
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS schema"
)


# Write the DataFrame to a new table
df.write.mode('overwrite').saveAsTable('schema.test1')

display(df)

###adding data into the schema 

In [0]:
# Define the data to be added
data = [
    ("I001", "Low Fat", 0.016047, "Dairy", 249.8092),
    ("I002", "Regular", 0.019278, "Soft Drinks", 48.2692),
    ("I003", "Low Fat", 0.016760, "Meat", 141.6180)
]

# Create a DataFrame with the new data
new_df = spark.createDataFrame(data, schema)

# Append the new data to the existing table
new_df.write.mode('append').saveAsTable('schema.test1')

# Display the updated table
df = spark.read.table('schema.test1')
display(df)

###Sorting two column

In [0]:
df=df.sort(['Itemid','Price'],ascending=[True,False]).display()


###Limiting no of column

In [0]:

df = spark.read.table('schema.test1')
df=df.limit(2).display()


In [0]:
# Define the new row to be added
new_row = [("I004", "Regular", 0.015, "Fruits", 120.50)]

# Create a DataFrame with the new row
new_row_df = spark.createDataFrame(new_row, schema)

# Append the new row to the existing table
new_row_df.write.mode('append').saveAsTable('schema.test1')

# Display the updated table
df = spark.read.table('schema.test1')
display(df)

###Droping a column

In [0]:
df=df.drop('visibility').display()

###Appending a new column to the table 

In [0]:
new_row = [("I004", "Regular", 0.015, "Fruits", 120.50)]

# Create a DataFrame with the new row
new_row_df = spark.createDataFrame(new_row, schema)

# Append the new row to the existing table
new_row_df.write.mode('append').saveAsTable('schema.test1')

# Display the updated table
df = spark.read.table('schema.test1')
display(df)

### Drop duplicates from table

In [0]:
df.dropDuplicates().display()

###Drop duplicates based on particular column 

In [0]:
df = spark.read.table('test.sales2')
display(df)

In [0]:
df = df.dropDuplicates(['Item_Visibility'])
display(df)

###Distinct column

In [0]:
from pyspark.sql.functions import col

df = spark.read.table('test.sales2')
df.distinct().display()

###Union and union by name 

In [0]:
data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
display(df1)

In [0]:
df1.union(df2).display()


In [0]:
data1 = [('kad','1'),
        ('sid','2')]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)
        

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

###String function

### Initcap

In [0]:
from pyspark.sql.functions import upper

df = spark.read.table('test.sales2')
display(
    df.select(
        upper('Item_type').alias('Item_type_upper')
    )
)

### Date function

#### Current date

In [0]:
from pyspark.sql.functions import current_date
df=df.withColumn('item_date',current_date())
display(df)

###Merging the schema 

In [0]:
from pyspark.sql.functions import current_date

df = spark.read.table('test.sales2')
df = df.withColumn(
    'item_date',
    current_date()
)
df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('test.sales2')
display(df)

#### Date_add

In [0]:
from pyspark.sql.functions import date_add

df = spark.read.table('test.sales2')
df = df.withColumn(
    'week_After',
    date_add('item_date', 7)
)
df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('test.sales2')
display(df)


#### Date_Sub

In [0]:
from pyspark.sql.functions import date_sub

df = spark.read.table('test.sales2')
df = df.withColumn(
    'week_before',
    date_sub('item_date', 7)
)
df.write.mode('overwrite').option('mergeSchema', 'true').saveAsTable('test.sales2')
display(df)

#### Date_diff

In [0]:
from pyspark.sql.functions import datediff

df = df.withColumn(
    'date_diff',
    datediff('week_After', 'item_date')
)
display(df)

### Date format 

In [0]:
from pyspark.sql.functions import date_format
df = df.withColumn('Item_date',date_format('week_before','dd-MM-yyyy'))

df.display()

### Drope null

In [0]:
df.dropna(subset=['Outlet_Size']).display()

### filling nulls

In [0]:
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable',subset=['Outlet_Size']).display()

### Split

In [0]:
from pyspark.sql.functions import split
df=spark.read.table("testing.sales2")
# Assuming df is already defined and loaded with data
df = df.withColumn('Outlet_Type', split(df['Outlet_Type'], ' '))
display(df)

###Indexing

In [0]:
from pyspark.sql.functions import split
df=spark.read.table("testing.sales2")
# Assuming df is already defined and loaded with data
df = df.withColumn('Outlet_Type', split(df['Outlet_Type'], ' ')[1])
display(df)

###Explode

In [0]:
from pyspark.sql.functions import split
df=spark.read.table("testing.sales2")
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

In [0]:
from pyspark.sql.functions import split, explode

df_exp = spark.read.table("testing.sales2")
df_exp = df_exp.withColumn('Outlet_Type_Array', split('Outlet_Type', ','))
df_exp = df_exp.withColumn('Outlet_Type', explode('Outlet_Type_Array')).drop('Outlet_Type_Array')
display(df_exp)

###Array_Contains

In [0]:
from pyspark.sql.functions import split, array_contains

# Split the 'Outlet_Type' column into an array
df_exp = df_exp.withColumn('Outlet_Type_Array', split('Outlet_Type', ' '))

# Use array_contains on the array column
df_exp = df_exp.withColumn('Type1_flag', array_contains('Outlet_Type_Array', 'Type1'))

# Display the DataFrame
display(df_exp)

###Groupby

In [0]:
from pyspark.sql.functions import sum
df=spark.read.table("testing.sales2")
df.groupBy("Item_Type").agg(sum("Item_price")).display()

In [0]:
from pyspark.sql.functions import sum,avg
df=spark.read.table("testing.sales2")
df.groupBy("Item_Type").agg(avg("Item_price")).display()

In [0]:
from pyspark.sql.functions import sum
df=spark.read.table("testing.sales2")
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_price").alias("Total_price")).display()

In [0]:
from pyspark.sql.functions import sum
df=spark.read.table("testing.sales2")
df.groupBy("Item_Type","Outlet_Size").agg(sum("Item_price").alias("Total_price"),avg("Item_price").alias("average_price")).display()

In [0]:
def fibonacci_series(n):
    a, b = 0, 1
    for _ in range(n):
        print(a, end=" ")
        a, b = b, a + b

# Example usage
fibonacci_series(10)  # Prints first 10 Fibonacci numbers

